# San Diego Neighborhood Opportunity Finder

## Final Dataset Merge and Cleaning

This notebook finishes the residential and mixed-use census tract dataset before exploratory analysis and scoring.

The main datasets were already cleaned and combined in earlier notebooks. This notebook focuses on the remaining issues that could affect later analysis:

- correcting missing walkability values caused by older EPA boundaries
- reviewing and estimating missing median gross rent values
- keeping original source values separate from estimated values
- tracking recalculated or estimated values with flags
- removing temporary helper columns
- confirming that each tract appears only once
- saving the corrected final dataset

The final output from this notebook will be used as the starting dataset for `08_eda.ipynb`.

This notebook does not create the final opportunity score or make final tract recommendations.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import geopandas as gpd

# prefered theme
sns.set_theme(style='whitegrid', palette='Set2')

# using this to show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
file_path = '../data/processed/master_residential_tract_features.csv'

df = pd.read_csv(
    file_path,
    dtype={'tract_id': str}) # keep census tract ID as text so it's not accidentally calculated for anything

## Dataset Overview

In [3]:
df.shape

(727, 72)

In [4]:
# previewing the main ID and feature columns without showing all 72 columns

preview_cols = [
    'tract_id',
    'tract_name',
    'total_population',
    'median_household_income',
    'median_gross_rent',
    'walkability_index',
    'transit_stop_density',
    'school_academic_score']

df[preview_cols].head()

,tract_id,tract_name,total_population,median_household_income,median_gross_rent,walkability_index,transit_stop_density,school_academic_score
0,06073000100,Census Tract 1; San Diego County; California,2948,231667.0,NaN,14.740561,10.115410,3.5
1,06073000201,Census Tract 2.01; San Diego County; California,2270,124722.0,2407.0,18.000000,20.978476,3.5
2,06073000202,Census Tract 2.02; San Diego County; California,3755,120091.0,1992.0,15.573006,21.814380,3.5
3,06073000301,Census Tract 3.01; San Diego County; California,2311,87813.0,1945.0,NaN,45.122430,3.5
4,06073000302,Census Tract 3.02; San Diego County; California,2873,89573.0,2412.0,NaN,52.115092,3.5


In [5]:
# checking the starting dataset before making final corrections

starting_checks = pd.Series({
    'rows': len(df),
    'columns': df.shape[1],
    'unique_tract_ids': df['tract_id'].nunique(),
    'duplicate_tract_ids': df['tract_id'].duplicated().sum(),
    'total_missing_values': df.isna().sum().sum()})

starting_checks

rows                     727
columns                   72
unique_tract_ids         727
duplicate_tract_ids        0
total_missing_values    1082
dtype: int64

## Final Walkability Correction

The main dataset is based on 2024 census tract boundaries, but the EPA walkability data uses 2020 boundaries.

There were multiple null walkability fields for 199 tracts, so I'm doing a spatial overlay to match the older EPA polygons to the 2024 tract boundaries and rebuild the missing values using household-weighted averages. I'm using this because walkability matters most where people actually live. Areas with more households should have more influence on the tract’s final walkability value than large areas with very few residents.

I’ll keep the original values as is and add a flag showing which tracts needed the correction.

In [6]:
# setting the walkability data path

raw_dir = Path('../data/raw/WalkabilityIndex')
walkability_path = raw_dir / 'Natl_WI.gdb'

In [7]:
# checking the available layers

gpd.list_layers(walkability_path)

,name,geometry_type
0,NationalWalkabilityIndex,MultiPolygon


In [8]:
# loading the walkability geography

walk_gdf = gpd.read_file(
    walkability_path,
    layer='NationalWalkabilityIndex')

walk_gdf.shape

(220739, 30)

In [9]:
# checking the walkability source before filtering to San Diego County

walk_source_checks = pd.Series({
    'rows': len(walk_gdf),
    'columns': walk_gdf.shape[1],
    'crs': str(walk_gdf.crs),
    'missing_geoid20': walk_gdf['GEOID20'].isna().sum()})

walk_source_checks

rows                                                          220739
columns                                                           30
crs                PROJCS["USA_Contiguous_Albers_Equal_Area_Conic...
missing_geoid20                                                    0
dtype: object

In [10]:
# filtering to San Diego County

sd_walk_gdf = walk_gdf[
    walk_gdf['GEOID20'].str.startswith('06073')].copy()

sd_walk_gdf.shape

(1795, 30)

In [11]:
tract_path = Path('../data/raw/census_tracts/tl_2024_06_tract.shp')

tract_gdf = gpd.read_file(tract_path)

tract_gdf.shape

(9129, 14)

In [12]:
# filtering to San Diego County tracts

sd_tract_gdf = tract_gdf[
    tract_gdf['COUNTYFP'] == '073'].copy()

sd_tract_gdf.shape

(737, 14)

In [13]:
sd_tract_gdf[['GEOID', 'NAME']].head()

,GEOID,NAME
817,06073008331,83.31
818,06073008336,83.36
819,06073008337,83.37
820,06073011601,116.01
821,06073011602,116.02


In [14]:
# checking the 2024 San Diego tract boundaries before the spatial overlay

tract_source_checks = pd.Series({
    'rows': len(sd_tract_gdf),
    'columns': sd_tract_gdf.shape[1],
    'unique_tract_ids': sd_tract_gdf['GEOID'].nunique(),
    'crs': str(sd_tract_gdf.crs)})

tract_source_checks

rows                      737
columns                    14
unique_tract_ids          737
crs                 EPSG:4269
dtype: object

In [15]:
# matching both layers to the same CRS before the spatial overlay

sd_tract_gdf = sd_tract_gdf.to_crs(
    sd_walk_gdf.crs)

sd_tract_gdf = sd_tract_gdf.rename(
    columns={'GEOID': 'tract_id'})

sd_tract_gdf.crs == sd_walk_gdf.crs

True

In [16]:
# keeping only the walkability fields needed for the spatial overlay

walk_cols = [
    'GEOID20',
    'HH',
    'NatWalkInd',
    'D2A_Ranked',
    'D2B_Ranked',
    'D3B_Ranked',
    'D4A_Ranked',
    'geometry']

walk_overlay_gdf = sd_walk_gdf[
    walk_cols].copy()

In [17]:
# saving each EPA polygon's full area before it gets split across tracts

walk_overlay_gdf['source_area'] = (walk_overlay_gdf.geometry.area)

In [18]:
walk_overlay_gdf.shape

(1795, 9)

In [19]:
# overlaying the EPA block groups with the 2024 tract boundaries

walk_tract_overlap = gpd.overlay(
    walk_overlay_gdf,
    sd_tract_gdf[['tract_id', 'geometry']],
    how='intersection')

walk_tract_overlap.shape

(7576, 10)

In [20]:
# calculating how much of each EPA polygon falls inside each tract

walk_tract_overlap['overlap_area'] = (
    walk_tract_overlap.geometry.area)

walk_tract_overlap['overlap_pct'] = (
    walk_tract_overlap['overlap_area']
    / walk_tract_overlap['source_area'])

walk_tract_overlap[
    ['tract_id', 'GEOID20', 'overlap_pct']].head()

,tract_id,GEOID20,overlap_pct
0,06073017069,060730170341,6.718603e-10
1,06073017071,060730170341,8.521232e-09
2,06073017033,060730170341,5.939257e-09
3,06073017018,060730170341,8.822476e-10
4,06073017034,060730170341,1.000000e+00


In [21]:
# removing tiny boundary slivers that shouldn't affect the tract average

walk_tract_overlap = walk_tract_overlap[
    walk_tract_overlap['overlap_pct'] >= 0.01].copy()

walk_tract_overlap.shape

(1926, 12)

Because the boundaries in the initial merge didn't line up exactly, I had to do a spatial overlay. It shows which older walkability areas fall inside each newer tract, then rebuild a walkability score for the 2024 tract. This was more accurate than imputing an average from nearby tracts.

<b>Process</b>

I selected the EPA walkability fields including household counts, then matched the EPA block-group polygons to the updated 2024 census tract boundaries. The overlay split polygons wherever the two boundary systems crossed. After that, I calculated how much of each EPA block group falls inside each 2024 tract. The summary shows a lot of slim overlaps, so we’ll need to remove those small slivers before calculating the final weighted walkability values.

In [22]:
# estimating how many households fall inside each tract overlap

walk_tract_overlap['adjusted_hh'] = (
    walk_tract_overlap['HH']
    * walk_tract_overlap['overlap_pct'])

walk_tract_overlap[
    ['tract_id', 'GEOID20', 'HH', 'overlap_pct', 'adjusted_hh']].sort_values(by='overlap_pct', ascending=False).head()

,tract_id,GEOID20,HH,overlap_pct,adjusted_hh
6018,06073010009,060730100092,466.0,1.0,466.0
7399,06073020304,060730203042,208.0,1.0,208.0
587,06073008501,060730085011,685.0,1.0,685.0
6257,06073016901,060730169013,427.0,1.0,427.0
5699,06073008362,060730083623,600.0,1.0,600.0


In [23]:
# calculating household-weighted walkability values by tract

walk_features = [
    'NatWalkInd',
    'D2A_Ranked',
    'D2B_Ranked',
    'D3B_Ranked',
    'D4A_Ranked']

for col in walk_features:
    walk_tract_overlap[f'{col}_weighted'] = (
        walk_tract_overlap[col]
        * walk_tract_overlap['adjusted_hh'])

In [24]:
# combining the overlap pieces into one row per tract

walk_2024 = (
    walk_tract_overlap
    .groupby('tract_id')
    .agg(
        total_adjusted_hh=('adjusted_hh', 'sum'),
        NatWalkInd_weighted=('NatWalkInd_weighted', 'sum'),
        D2A_Ranked_weighted=('D2A_Ranked_weighted', 'sum'),
        D2B_Ranked_weighted=('D2B_Ranked_weighted', 'sum'),
        D3B_Ranked_weighted=('D3B_Ranked_weighted', 'sum'),
        D4A_Ranked_weighted=('D4A_Ranked_weighted', 'sum'))
    .reset_index())

walk_2024.shape

(737, 7)

This grouped all of the smaller overlap pieces back into one row for each 2024 census tract. For every tract, it added up the estimated households and the weighted walkability values. Now there are 737 rows, which matches the number of San Diego County tracts. I still need to divide each weighted total by the total adjusted households to get the final average walkability scores.

In [25]:
# calculating the final average walkability values for each tract

for col in walk_features:
    walk_2024[col] = (
        walk_2024[f'{col}_weighted']
        / walk_2024['total_adjusted_hh'])

walk_2024[
    [
        'tract_id',
        'NatWalkInd',
        'D2A_Ranked',
        'D2B_Ranked',
        'D3B_Ranked',
        'D4A_Ranked'
    ]].head()

,tract_id,NatWalkInd,D2A_Ranked,D2B_Ranked,D3B_Ranked,D4A_Ranked
0,06073000100,14.740238,5.234290,10.738557,17.558573,18.675718
1,06073000201,18.000000,16.000000,20.000000,17.000000,19.000000
2,06073000202,15.573006,9.379487,14.026923,17.648718,17.367094
3,06073000301,14.547884,8.397322,9.512863,16.000000,18.688559
4,06073000302,16.098350,14.517801,9.930760,17.442156,18.628614


In [26]:
# renaming the recalculated walkability fields

walk_2024 = walk_2024.rename(
    columns={
        'NatWalkInd': 'walkability_index_new',
        'D2A_Ranked': 'jobs_housing_mix_score_new',
        'D2B_Ranked': 'employment_mix_score_new',
        'D3B_Ranked': 'intersection_density_score_new',
        'D4A_Ranked': 'commute_mode_diversity_score_new'})

walk_2024[
    [
        'tract_id',
        'walkability_index_new',
        'jobs_housing_mix_score_new',
        'employment_mix_score_new',
        'intersection_density_score_new',
        'commute_mode_diversity_score_new']].head()

,tract_id,walkability_index_new,jobs_housing_mix_score_new,employment_mix_score_new,intersection_density_score_new,commute_mode_diversity_score_new
0,06073000100,14.740238,5.234290,10.738557,17.558573,18.675718
1,06073000201,18.000000,16.000000,20.000000,17.000000,19.000000
2,06073000202,15.573006,9.379487,14.026923,17.648718,17.367094
3,06073000301,14.547884,8.397322,9.512863,16.000000,18.688559
4,06073000302,16.098350,14.517801,9.930760,17.442156,18.628614


In [27]:
# merging the recalculated walkability values into the residential dataset

walk_new_cols = [
    'tract_id',
    'walkability_index_new',
    'jobs_housing_mix_score_new',
    'employment_mix_score_new',
    'intersection_density_score_new',
    'commute_mode_diversity_score_new']

df = df.merge(
    walk_2024[walk_new_cols],
    on='tract_id',
    how='left')

df.shape

(727, 77)

In [28]:
df[[
    'tract_id',
    'walkability_index',
    'walkability_index_new',
    'missing_walkability_flag']].head()

,tract_id,walkability_index,walkability_index_new,missing_walkability_flag
0,06073000100,14.740561,14.740238,0
1,06073000201,18.000000,18.000000,0
2,06073000202,15.573006,15.573006,0
3,06073000301,NaN,14.547884,1
4,06073000302,NaN,16.098350,1


In [29]:
# filling only the missing walkability values

walk_pairs = {
    'walkability_index': 'walkability_index_new',
    'jobs_housing_mix_score': 'jobs_housing_mix_score_new',
    'employment_mix_score': 'employment_mix_score_new',
    'intersection_density_score': 'intersection_density_score_new',
    'commute_mode_diversity_score': 'commute_mode_diversity_score_new'}

for old_col, new_col in walk_pairs.items():
    df[old_col] = df[old_col].fillna(df[new_col])

df[list(walk_pairs.keys())].isna().sum()

walkability_index               0
jobs_housing_mix_score          0
employment_mix_score            0
intersection_density_score      0
commute_mode_diversity_score    0
dtype: int64

In [30]:
# marking tracts where walkability values were recalculated

df['walkability_imputed_flag'] = df['missing_walkability_flag']

df['walkability_imputed_flag'].value_counts()

walkability_imputed_flag
0    528
1    199
Name: count, dtype: int64

## Walkability Null Values

In retrospect, I should have done this step in the data wrangling phase when I saw that there were a lot of null values.

The missing walkability values came from a mismatch between the older EPA block-group boundaries and the newer 2024 census tract boundaries. I used a spatial overlay to match the two boundary types and recalculate walkability values for each 2024 tract.

In this process, I filled 199 missing tract values. I also kept a flag showing which values were recalculated so I can track them later during scoring

## Final Rent Imputation

After fixing the walkability fields, the main remaining gaps are in the original ACS rent and household income columns.

I’ll keep the original Census columns as is, but make a separate estimated rent field for analysis and keep a flag showing which rent values were originally missing.

In [31]:
# checking the remaining null values

null_summary = (
    df.isna()
    .sum()
    .to_frame('missing_count'))

null_summary['missing_pct'] = (
    null_summary['missing_count'] / len(df) * 100).round(2)

null_summary = (
    null_summary[
        null_summary['missing_count'] > 0]
    .sort_values('missing_pct', ascending=False))

null_summary

,missing_count,missing_pct
median_gross_rent,79,10.87
median_household_income,8,1.10


## Missing Median Gross Rent

`median_gross_rent` is missing for 79 tracts, or about 10.9% of the residential and mixed-use dataset.

Before estimating those values, I want to check whether the missing rent is connected to low renter rates, income, poverty, vacancy, or other factors.

I think I'll leave the household income null since it's close to 1% null values, but I'll address median rent because I might want to address rental properties for development. 

In [32]:
# reviewing tracts with missing rent

missing_rent = df[
    df['median_gross_rent'].isna()].copy()

missing_rent[
    [
        'tract_id',
        'tract_name',
        'total_population',
        'total_households',
        'renter_rate',
        'median_household_income',
        'poverty_rate',
        'vacancy_rate',
        'tract_type_flag']].head(20)

,tract_id,tract_name,total_population,total_households,renter_rate,median_household_income,poverty_rate,vacancy_rate,tract_type_flag
0,06073000100,Census Tract 1; San Diego County; California,2948,1178,9.4,231667.0,2.2,8.9,residential_or_mixed
123,06073006600,Census Tract 66; San Diego County; California,2032,477,100.0,110651.0,6.0,26.6,residential_or_mixed
127,06073007002,Census Tract 70.02; San Diego County; California,3025,1256,11.8,184167.0,4.6,3.2,residential_or_mixed
130,06073007302,Census Tract 73.02; San Diego County; California,2610,984,20.7,183529.0,2.1,4.6,residential_or_mixed
154,06073008202,Census Tract 82.02; San Diego County; California,1203,659,63.7,111836.0,3.9,38.3,residential_or_mixed
155,06073008301,Census Tract 83.01; San Diego County; California,3190,1347,16.9,216823.0,2.4,0.0,residential_or_mixed
156,06073008303,Census Tract 83.03; San Diego County; California,3055,1341,20.1,230511.0,6.1,19.7,residential_or_mixed
158,06073008306,Census Tract 83.06; San Diego County; California,2711,1092,8.0,204779.0,9.1,2.1,residential_or_mixed
160,06073008310,Census Tract 83.10; San Diego County; California,6538,2457,7.5,172399.0,2.4,4.2,residential_or_mixed
161,06073008311,Census Tract 83.11; San Diego County; California,2739,1088,4.7,NaN,3.3,9.0,residential_or_mixed


In [33]:
# summarizing the missing-rent tracts

missing_rent[
    [
        'total_population',
        'total_households',
        'renter_rate',
        'median_household_income',
        'poverty_rate',
        'vacancy_rate']].describe().round(2)

,total_population,total_households,renter_rate,median_household_income,poverty_rate,vacancy_rate
count,79.00,79.00,79.00,71.00,79.00,79.00
mean,3925.77,1373.42,22.56,175152.49,4.85,8.04
std,1569.46,556.62,23.22,44067.22,3.74,8.59
min,1203.00,477.00,2.30,60455.00,0.60,0.00
25%,2725.00,998.50,7.85,152982.00,2.35,2.05
50%,3663.00,1256.00,14.10,183529.00,4.20,5.60
75%,4689.50,1593.00,25.90,208977.00,6.05,9.70
max,8172.00,2938.00,100.00,247222.00,28.20,38.30


It looks like rent is missing in higher-income, primarily owner-occupied tracts with less rental households. The missing values are probably not random, so I have to figure out how to impute. 

In [34]:
# comparing tracts with and without observed rent

df['rent_missing_flag'] = df['median_gross_rent'].isna().astype(int)

rent_missing_compare = (
    df.groupby('rent_missing_flag')[
        [
            'renter_rate',
            'median_household_income',
            'poverty_rate',
            'vacancy_rate',
            'total_population',
            'total_households']].median().round(2))

rent_missing_compare

,renter_rate,median_household_income,poverty_rate,vacancy_rate,total_population,total_households
rent_missing_flag,,,,,,
0,45.2,104849.5,8.7,4.6,4344.5,1582.0
1,14.1,183529.0,4.2,5.6,3663.0,1256.0


In [35]:
# checking which features are most related to observed rent

rent_relationships = (
    df[
        [
            'median_gross_rent',
            'median_household_income',
            'renter_rate',
            'poverty_rate',
            'vacancy_rate'
        ]
    ]
    .corr()['median_gross_rent']
    .drop('median_gross_rent')
    .sort_values(ascending=False))

rent_relationships

median_household_income    0.632264
vacancy_rate              -0.029759
renter_rate               -0.310573
poverty_rate              -0.431376
Name: median_gross_rent, dtype: float64

Income has the strongest relationship with rent (0.63). Poverty also has a moderate negative relationship (-0.43). Renter rate and vacancy rate are weaker so I can't really infer anything. 

In [36]:
# creating income groups from tracts with observed income

df['income_group'] = pd.qcut(
    df['median_household_income'],
    q=5,
    duplicates='drop')

df.groupby('income_group', observed=False)[
    'median_gross_rent'].median().round(2)

income_group
(34147.999, 76547.4]    1839.5
(76547.4, 98557.2]      2073.0
(98557.2, 118568.0]     2340.0
(118568.0, 146139.0]    2606.0
(146139.0, 247222.0]    2941.5
Name: median_gross_rent, dtype: float64

In [37]:
# creating a separate estimated rent column

income_group_rent = (
    df.groupby('income_group', observed=False)['median_gross_rent']
    .transform('median'))

df['estimated_median_gross_rent'] = (
    df['median_gross_rent']
    .fillna(income_group_rent))

df[
    [
        'median_gross_rent',
        'estimated_median_gross_rent'
    ]].isna().sum()

median_gross_rent              79
estimated_median_gross_rent     8
dtype: int64

In [38]:
# checking why some estimated rent values are still missing

df[
    df['estimated_median_gross_rent'].isna()
][
    [
        'tract_id',
        'tract_name',
        'median_household_income',
        'median_gross_rent',
        'renter_rate',
        'poverty_rate',
        'vacancy_rate'
    ]
].sort_values(by='renter_rate', ascending=False)

,tract_id,tract_name,median_household_income,median_gross_rent,renter_rate,poverty_rate,vacancy_rate
200,06073008372,Census Tract 83.72; San Diego County; California,NaN,NaN,43.5,7.1,4.9
245,06073009504,Census Tract 95.04; San Diego County; California,NaN,NaN,21.4,3.2,10.0
719,06073021501,Census Tract 215.01; San Diego County; California,NaN,NaN,19.3,2.2,5.0
202,06073008374,Census Tract 83.74; San Diego County; California,NaN,NaN,11.0,1.6,0.0
166,06073008328,Census Tract 83.28; San Diego County; California,NaN,NaN,10.4,4.9,13.6
498,06073017062,Census Tract 170.62; San Diego County; California,NaN,NaN,7.7,1.4,18.7
161,06073008311,Census Tract 83.11; San Diego County; California,NaN,NaN,4.7,3.3,9.0
514,06073017112,Census Tract 171.12; San Diego County; California,NaN,NaN,2.3,1.7,6.7


I'm interested in the tracts with higher rental rates (tracts 83.72, 95.04, and 215.01). I'll pull 3 separate tables to inspect those general tracks and see if there's a way to impute an average.

## Track 83

In [39]:
# reviewing tract 83 areas

tract_83_table = df[
    df['tract_name'].str.contains(
        r'Census Tract 83\.',
        regex=True,
        na=False
    )][
    [
        'tract_id',
        'tract_name',
        'median_household_income',
        'median_gross_rent',
        'estimated_median_gross_rent',
        'renter_rate',
        'poverty_rate',
        'vacancy_rate',
        'total_households'
    ]].sort_values('tract_name')

tract_83_table

,tract_id,tract_name,median_household_income,median_gross_rent,estimated_median_gross_rent,renter_rate,poverty_rate,vacancy_rate,total_households
155,06073008301,Census Tract 83.01; San Diego County; California,216823.0,NaN,2941.5,16.9,2.4,0.0,1347
156,06073008303,Census Tract 83.03; San Diego County; California,230511.0,NaN,2941.5,20.1,6.1,19.7,1341
157,06073008305,Census Tract 83.05; San Diego County; California,41840.0,1669.0,1669.0,97.7,27.1,3.1,471
158,06073008306,Census Tract 83.06; San Diego County; California,204779.0,NaN,2941.5,8.0,9.1,2.1,1092
159,06073008307,Census Tract 83.07; San Diego County; California,108667.0,2995.0,2995.0,21.0,8.2,6.6,1664
160,06073008310,Census Tract 83.10; San Diego County; California,172399.0,NaN,2941.5,7.5,2.4,4.2,2457
161,06073008311,Census Tract 83.11; San Diego County; California,NaN,NaN,NaN,4.7,3.3,9.0,1088
162,06073008312,Census Tract 83.12; San Diego County; California,143847.0,2384.0,2384.0,30.6,8.4,12.7,1677
163,06073008313,Census Tract 83.13; San Diego County; California,221536.0,NaN,2941.5,13.8,7.3,10.4,809
164,06073008324,Census Tract 83.24; San Diego County; California,197535.0,NaN,2941.5,20.4,6.0,6.9,2938


In [40]:
tract_83_table.info()

<class 'pandas.DataFrame'>
RangeIndex: 55 entries, 155 to 209
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   tract_id                     55 non-null     str    
 1   tract_name                   55 non-null     str    
 2   median_household_income      51 non-null     float64
 3   median_gross_rent            36 non-null     float64
 4   estimated_median_gross_rent  51 non-null     float64
 5   renter_rate                  55 non-null     float64
 6   poverty_rate                 55 non-null     float64
 7   vacancy_rate                 55 non-null     float64
 8   total_households             55 non-null     int64  
dtypes: float64(6), int64(1), str(2)
memory usage: 4.0 KB


In [41]:
tract_83_table.describe()

,median_household_income,median_gross_rent,estimated_median_gross_rent,renter_rate,poverty_rate,vacancy_rate,total_households
count,51.000000,36.000000,51.000000,55.000000,55.000000,55.000000,55.000000
mean,147732.176471,2751.944444,2807.696078,42.694545,10.530909,6.216364,1614.454545
std,49183.954560,451.595517,387.770474,26.835427,10.612028,5.208567,607.327077
min,41840.000000,1618.000000,1618.000000,4.700000,1.300000,0.000000,391.000000
25%,109646.000000,2483.750000,2626.000000,20.300000,3.550000,2.850000,1145.500000
50%,149052.000000,2759.000000,2941.500000,42.900000,7.000000,5.300000,1582.000000
75%,192132.500000,3102.500000,2941.500000,63.600000,9.450000,8.300000,1969.500000
max,230956.000000,3427.000000,3427.000000,97.700000,44.500000,24.800000,2938.000000


## Tract 95

In [42]:
# reviewing tract 95 areas

tract_95_table = df[
    df['tract_name'].str.contains(
        r'Census Tract 95\.',
        regex=True,
        na=False)][
    [
        'tract_id',
        'tract_name',
        'median_household_income',
        'median_gross_rent',
        'estimated_median_gross_rent',
        'renter_rate',
        'poverty_rate',
        'vacancy_rate',
        'total_households'
    ]].sort_values('median_gross_rent')

tract_95_table

,tract_id,tract_name,median_household_income,median_gross_rent,estimated_median_gross_rent,renter_rate,poverty_rate,vacancy_rate,total_households
244,06073009502,Census Tract 95.02; San Diego County; California,141786.0,2601.0,2601.0,32.7,3.3,4.1,1552
248,06073009507,Census Tract 95.07; San Diego County; California,117330.0,2789.0,2789.0,42.2,5.4,7.3,1503
246,06073009505,Census Tract 95.05; San Diego County; California,121960.0,3005.0,3005.0,30.7,1.9,3.1,2809
247,06073009506,Census Tract 95.06; San Diego County; California,160956.0,3200.0,3200.0,25.0,6.4,1.3,1586
249,06073009509,Census Tract 95.09; San Diego County; California,148568.0,3215.0,3215.0,80.2,5.9,9.0,2521
245,06073009504,Census Tract 95.04; San Diego County; California,NaN,NaN,NaN,21.4,3.2,10.0,2356
250,06073009510,Census Tract 95.10; San Diego County; California,60497.0,NaN,1839.5,97.9,12.7,21.3,1168
251,06073009511,Census Tract 95.11; San Diego County; California,78102.0,NaN,2073.0,98.4,10.2,14.1,971


In [43]:
tract_95_table.describe()

,median_household_income,median_gross_rent,estimated_median_gross_rent,renter_rate,poverty_rate,vacancy_rate,total_households
count,7.000000,5.00000,7.000000,8.000000,8.000000,8.000000,8.000000
mean,118457.000000,2962.00000,2674.642857,53.562500,6.125000,8.775000,1808.250000
std,37100.108549,265.87215,540.841991,33.000127,3.681518,6.534469,668.547199
min,60497.000000,2601.00000,1839.500000,21.400000,1.900000,1.300000,971.000000
25%,97716.000000,2789.00000,2337.000000,29.275000,3.275000,3.850000,1419.250000
50%,121960.000000,3005.00000,2789.000000,37.450000,5.650000,8.150000,1569.000000
75%,145177.000000,3200.00000,3102.500000,84.625000,7.350000,11.025000,2397.250000
max,160956.000000,3215.00000,3215.000000,98.400000,12.700000,21.300000,2809.000000


In [44]:
# reviewing tract 215 areas

tract_215_table = df[
    df['tract_name'].str.contains(
        r'Census Tract 215\.',
        regex=True,
        na=False
    )][
    [
        'tract_id',
        'tract_name',
        'median_household_income',
        'median_gross_rent',
        'estimated_median_gross_rent',
        'renter_rate',
        'poverty_rate',
        'vacancy_rate',
        'total_households'
    ]].sort_values('median_gross_rent', ascending=False)

tract_215_table

,tract_id,tract_name,median_household_income,median_gross_rent,estimated_median_gross_rent,renter_rate,poverty_rate,vacancy_rate,total_households
720,06073021502,Census Tract 215.02; San Diego County; California,181644.0,3233.0,3233.0,49.8,3.8,7.3,3231
719,06073021501,Census Tract 215.01; San Diego County; California,NaN,NaN,NaN,19.3,2.2,5.0,1225


In [45]:
tract_215_table.describe()

,median_household_income,median_gross_rent,estimated_median_gross_rent,renter_rate,poverty_rate,vacancy_rate,total_households
count,1.0,1.0,1.0,2.000000,2.000000,2.000000,2.000000
mean,181644.0,3233.0,3233.0,34.550000,3.000000,6.150000,2228.000000
std,NaN,NaN,NaN,21.566757,1.131371,1.626346,1418.456203
min,181644.0,3233.0,3233.0,19.300000,2.200000,5.000000,1225.000000
25%,181644.0,3233.0,3233.0,26.925000,2.600000,5.575000,1726.500000
50%,181644.0,3233.0,3233.0,34.550000,3.000000,6.150000,2228.000000
75%,181644.0,3233.0,3233.0,42.175000,3.400000,6.725000,2729.500000
max,181644.0,3233.0,3233.0,49.800000,3.800000,7.300000,3231.000000


In [46]:
# getting tract centers for distance checks

rent_geo = sd_tract_gdf[
    ['tract_id', 'geometry']].copy() # making a small geoDF with tract id and geometry

rent_geo['centroid'] = rent_geo.geometry.centroid # using this to find the center of the tracts' geometry
rent_geo = rent_geo.set_geometry('centroid') # use the centerpoint instead of the tract boundaries

# using these columns for the centerpoints, and making sure to keep every tract in rent_geo, even if some fields from df are missing.

rent_geo = rent_geo.merge(
    df[
        [
            'tract_id',
            'tract_name',
            'median_gross_rent',
            'renter_rate',
            'poverty_rate'
        ]
    ],
    on='tract_id',
    how='left')

In [47]:
target_tracts = [
    '06073008372',
    '06073009504',
    '06073021501',
    '06073008311', 
    '06073008328', 
    '06073008374', 
    '06073017062', 
    '06073017112']

In [48]:
# finding the five closest tracts with rent values

nearby_rent_tables = {}

for target_id in target_tracts:
    target_point = rent_geo.loc[
        rent_geo['tract_id'] == target_id,
        'centroid'].iloc[0] # getting the centerpoint and the first matching point

    nearby = rent_geo[
        rent_geo['median_gross_rent'].notna()       # only pulling tracts with data
        & (rent_geo['tract_id'] != target_id)].copy() # not pulling the tract in question

    nearby['distance_miles'] = (
        nearby.geometry.distance(target_point) / 1609.34) # conversion from meters to miles

    nearby_rent_tables[target_id] = (
        nearby[
            [
                'tract_id',
                'tract_name',
                'median_gross_rent',
                'renter_rate',
                'poverty_rate',
                'distance_miles'
            ]]
        .sort_values('distance_miles')
        .head(5))

In [49]:
# showing each target tract's nearby rent table separately

for target_id in target_tracts:
    print(f'\nTarget tract: {target_id}')
    display(
        nearby_rent_tables[target_id].describe())


Target tract: 06073008372


,median_gross_rent,renter_rate,poverty_rate,distance_miles
count,5.000000,5.00000,5.000000,5.000000
mean,3147.200000,45.40000,5.060000,2.012705
std,298.124471,18.76566,2.694995,0.215388
min,2627.000000,28.70000,3.500000,1.765283
25%,3197.000000,30.50000,3.500000,1.857039
50%,3233.000000,42.90000,3.800000,2.001579
75%,3316.000000,49.80000,4.700000,2.135528
max,3363.000000,75.10000,9.800000,2.304095



Target tract: 06073009504


,median_gross_rent,renter_rate,poverty_rate,distance_miles
count,5.000000,5.00000,5.000000,5.000000
mean,2352.600000,28.64000,6.020000,3.255173
std,511.730202,15.38873,1.449828,0.330931
min,1563.000000,10.50000,4.200000,2.778955
25%,2184.000000,25.60000,4.900000,3.074796
50%,2452.000000,26.80000,6.200000,3.392265
75%,2690.000000,27.10000,7.400000,3.403106
max,2874.000000,53.20000,7.400000,3.626740



Target tract: 06073021501


...

,median_gross_rent,renter_rate,poverty_rate,distance_miles
count,5.000000,5.000000,5.000000,5.000000
mean,3291.000000,54.980000,5.280000,1.609769
std,77.913413,13.890536,2.682722,0.305655
min,3197.000000,42.900000,3.500000,1.323756
25%,3233.000000,43.800000,3.600000,1.438479
50%,3288.000000,49.800000,3.800000,1.472770
75%,3363.000000,63.300000,5.700000,1.723635
max,3374.000000,75.100000,9.800000,2.090207



Target tract: 06073017062


,median_gross_rent,renter_rate,poverty_rate,distance_miles
count,5.000000,5.000000,5.000000,5.000000
mean,2671.600000,31.100000,3.820000,2.686503
std,829.051446,18.458467,2.176465,0.674476
min,1618.000000,12.900000,1.300000,1.703625
25%,1930.000000,12.900000,3.200000,2.468431
50%,3208.000000,30.100000,3.500000,2.696023
75%,3233.000000,49.800000,3.800000,3.062878
max,3369.000000,49.800000,7.300000,3.501559



Target tract: 06073017112


,median_gross_rent,renter_rate,poverty_rate,distance_miles
count,5.000000,5.000000,5.000000,5.000000
mean,2354.000000,22.440000,3.240000,2.375817
std,379.588593,15.642506,1.879628,0.655258
min,1754.000000,8.000000,1.200000,1.457418
25%,2381.000000,8.500000,1.700000,1.984785
50%,2392.000000,21.700000,3.000000,2.620685
75%,2433.000000,28.300000,4.800000,2.690747
max,2810.000000,45.700000,5.500000,3.125448


    '06073008372'
    '06073009504'
    '06073021501'
    
These 3 tracts had both income and rent were missing, so I couldn't base anything off of income groups. I used each tract’s center point to find the five closest tracts with actual rent data. I looked at the nearby rent and used the median for an estimate.

In [50]:
# calculating the median nearby rent for each target tract

target_rent_estimates = {
    target_id: nearby_rent_tables[target_id][
        'median_gross_rent'
    ].median()
    for target_id in target_tracts}

pd.Series(
    target_rent_estimates,
    name='estimated_median_gross_rent').to_frame()

,estimated_median_gross_rent
06073008372,3233.0
06073009504,2452.0
06073021501,3288.0
06073008311,2806.0
06073008328,3233.0
06073008374,3288.0
06073017062,3208.0
06073017112,2392.0


I used a loop instead of manually inputing values to make it more scalable. Originally I started to inpute manually, but it seemed inefficient.

In [51]:
# filling the remaining rent estimates

for tract_id, rent_value in target_rent_estimates.items():
    df.loc[
        (df['tract_id'] == tract_id)
        & (df['estimated_median_gross_rent'].isna()),
        'estimated_median_gross_rent'
    ] = rent_value

In [52]:
# checking that estimated rent is complete

df['estimated_median_gross_rent'].isna().sum()

np.int64(0)

## Reviewing the df and null values

In [53]:
# checking remaining null values after rent estimation

null_summary = (
    df.isna()
    .sum()
    .to_frame('missing_count'))

null_summary['missing_pct'] = (
    null_summary['missing_count'] / len(df) * 100).round(2)

null_summary = (
    null_summary[
        null_summary['missing_count'] > 0
    ]
    .sort_values('missing_pct', ascending=False))

null_summary

,missing_count,missing_pct
median_gross_rent,79,10.87
median_household_income,8,1.10
income_group,8,1.10


In [54]:
# checking original and estimated rent values

df[
    [
        'median_gross_rent',
        'estimated_median_gross_rent']].isna().sum()

median_gross_rent              79
estimated_median_gross_rent     0
dtype: int64

## Next Steps: 

    1. Clean up temporary columns (income_group, _new walkability columns)
    2. Confirm shape and number of tracts
    3. Organize features into groups (demographics, housing, walkability, transit, climate, schools)
    4. Review distributions (look at outliers, skew, and if I need to scale)
    5. Compare relationships between key features
    6. Look at tradeoffs (walkability vs safety, school strength vs rent, etc)
    7. Decide how to do scoring

## Final Data Quality Check

After handling the missing walkability and rent values, I cleaned up the temporary helper columns and checked the dataset again.

The original ACS rent and income columns still keep their missing values. For later analysis, I’ll use the separate estimated rent column when needed.

In [55]:
# removing temporary columns

helper_cols = [
    'income_group',
    'walkability_index_new',
    'jobs_housing_mix_score_new',
    'employment_mix_score_new',
    'intersection_density_score_new',
    'commute_mode_diversity_score_new']

df = df.drop(
    columns=[
        col for col in helper_cols
        if col in df.columns])

df.shape

(727, 75)

In [56]:
# checking rows, tract IDs, duplicates, and remaining nulls

print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')
print(f'Unique tracts: {df["tract_id"].nunique()}')
print(f'Duplicate tract IDs: {df["tract_id"].duplicated().sum()}')

df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)

Rows: 727
Columns: 75
Unique tracts: 727
Duplicate tract IDs: 0


median_gross_rent          79
median_household_income     8
dtype: int64

The final dataset has one row for each residential/mixed-use tract.

The original Census income and rent columns are still missing some values, but I determined that they would't be too impactful in scoring. I kept those columns unchanged so observed and estimated values don’t get mixed together.

The estimated rent column is complete for all tracts.

In [57]:
# renaming the rent flag so it makes more sense now that all values have been imputed

df = df.rename(
    columns={
        'rent_missing_flag': 'original_rent_missing_flag'
    }
)

In [58]:
# checking the final dataset before saving it

walkability_cols = [
    'walkability_index',
    'jobs_housing_mix_score',
    'employment_mix_score',
    'intersection_density_score',
    'commute_mode_diversity_score'
]

final_checks = pd.Series({
    'rows': len(df),
    'columns': df.shape[1],
    'unique_tract_ids': df['tract_id'].nunique(),
    'duplicate_tract_ids': df['tract_id'].duplicated().sum(),
    'missing_estimated_rent': df['estimated_median_gross_rent'].isna().sum(),
    'missing_active_walkability': df[walkability_cols].isna().sum().sum(),
    'walkability_imputed_tracts': df['walkability_imputed_flag'].sum(),
    'original_rent_missing_tracts': df['original_rent_missing_flag'].sum()
})

final_checks

rows                            727
columns                          75
unique_tract_ids                727
duplicate_tract_ids               0
missing_estimated_rent            0
missing_active_walkability        0
walkability_imputed_tracts      199
original_rent_missing_tracts     79
dtype: int64

In [59]:
# saving the corrected residential tract dataset for the EDA notebook

output_path = Path(
    '../data/processed/master_residential_tract_features_final.csv')

df.to_csv(
    output_path,
    index=False)

output_path

WindowsPath('../data/processed/master_residential_tract_features_final.csv')

## Final Merge Summary

I ended up having two final merge notebooks -- the first merge focused on combining the cleaned source datasets into one residential and mixed-use tract table. After starting the EDA, I found a few issues that still needed more work.

The biggest issue was the missing walkability data. I originally moved forward without fully resolving why 199 tracts were missing values. The problem came from using older boundaries with newer 2024 census tract boundaries. I corrected this with a spatial overlay and household-weighted averages.

I also found that missing rent values weren’t random. They were more common in higher-income, mostly owner-occupied tracts. I kept the original Census rent values and created a separate estimated rent field using income groups and nearby tract values.

Next time, I’d stop and investigate large groups of missing values earlier instead of carrying them into EDA. I’d also verify boundary years and geographic coverage before merging spatial datasets. The final dataset now has 727 unique tracts, complete active walkability fields, and complete estimated rent values.

In [ ]:
# grouping columns so the EDA is easier to manage

demographic_housing_features = [
    'median_household_income',
    'poverty_rate',
    'family_poverty_rate',
    'unemployment_rate',
    'renter_rate',
    'estimated_median_gross_rent',
    'vacancy_rate',
    'avg_household_size',
    'population_under_18_rate']

safety_features = [
    'safety_score',
    'violent_safety_score',
    'property_safety_score',
    'crime_rate_per_1000',
    'violent_crime_rate_per_1000',
    'property_crime_rate_per_1000']

walkability_transit_features = [
    'walkability_index',
    'jobs_housing_mix_score',
    'employment_mix_score',
    'intersection_density_score',
    'commute_mode_diversity_score',
    'transit_stop_density',
    'public_transit_commute_rate',
    'no_vehicle_rate']

environment_features = [
    'climate_loss_risk_score',
    'social_vulnerability_score',
    'community_resilience_score',
    'heat_risk_score',
    'flood_risk_score',
    'wildfire_risk_score']

school_features = [
    'school_density',
    'school_academic_score',
    'school_count',
    'elementary_school_count',
    'middle_school_count',
    'high_school_count']

In [ ]:
# selecting the main features to review first

core_eda_features = [
    'safety_score',
    'walkability_index',
    'transit_stop_density',
    'climate_loss_risk_score',
    'school_academic_score',
    'school_density',
    'median_household_income',
    'estimated_median_gross_rent',
    'poverty_rate',
    'renter_rate']

df[core_eda_features].describe().T.round(2)

In [ ]:
# checking whether the existing score fields actually vary

existing_score_cols = [
    'safety_score',
    'violent_safety_score',
    'property_safety_score',
    'climate_loss_risk_score',
    'social_vulnerability_score',
    'heat_risk_score',
    'flood_risk_score',
    'wildfire_risk_score',
    'school_academic_score']

df[existing_score_cols].agg(
    ['count', 'nunique', 'min', 'median', 'max', 'std']).T.round(2)

## Score Reference

- ` safety_score `: Overall safety score from the crime/safety notebook. It combines crime patterns into one general safety measure. Higher is better.

- ` violent_safety_score `: Safety score focused on violent crime from the crime/safety notebook. Higher is better.

- ` property_safety_score `: Safety score focused on property crime from the crime/safety notebook. Higher is better.

- ` climate_loss_risk_score `: expected annual loss risk from the environmental risk data. This was renamed from `expected_annual_loss_score_composite`. Lower is better.

- ` social_vulnerability_score `: Social vulnerability score from the environmental risk data. It gives an idea of how vulnerable a tract’s population could be during hazards or stress events. Lower is better.

- ` heat_risk_score `: Heat hazard risk score from the environmental risk data. Lower is better.

- ` flood_risk_score `: Inland flood hazard risk score from the environmental risk data. Lower is better.

- ` wildfire_risk_score `: Wildfire hazard risk score from the environmental risk data. Lower is better.

- ` school_academic_score `: Simple school strength score from the schools notebook, based on available school performance fields. Higher is better.

Safety and school scores are positive features, so higher is better. Environmental risk scores are risk features, so lower is better. If I use risk scores later, I’ll probably need to reverse them before combining them into one opportunity score.

In [ ]:
# selecting the main features for relationship checks

relationship_features = [
    'safety_score',
    'walkability_index',
    'transit_stop_density',
    'climate_loss_risk_score',
    'social_vulnerability_score',
    'school_academic_score',
    'school_density',
    'median_household_income',
    'estimated_median_gross_rent',
    'poverty_rate',
    'renter_rate',
    'vacancy_rate']

relationship_corr = (
    df[relationship_features]
    .corr()
    .round(2))

In [ ]:
# plotting relationships between the main EDA features

plt.figure(figsize=(11, 8))

sns.heatmap(
    relationship_corr,
    annot=True,
    fmt='.2f',
    center=0,
    cmap='coolwarm')

plt.title('Relationships Between Main Neighborhood Features')
plt.tight_layout()
plt.show()

<b>Relationships:</b>

    - Walkability and transit density go hand in hand. Both are higher in areas with more renters. Makes sense for urban areas.
    - Income is strongly related to rent and negatively related to poverty and renter rate (kind of obvious)
    - Safety has a moderate negative relationship with walkability, transit, and renter rate, which could point to an important tradeoff later.

So far, I don't think I've learned anything completely new, but there's at least data backed proof to my existing assumptions.

## Feature Distributions

Next, I’ll review the distributions of the main neighborhood features.

This helps show whether the values are evenly spread, heavily skewed, or affected by outliers. These patterns will matter later when I scale the variables and build the final scoring method.

In [ ]:
# reviewing distributions for the main access and risk features

distribution_features = [
    'walkability_index',
    'transit_stop_density',
    'climate_loss_risk_score',
    'social_vulnerability_score',
    'school_academic_score',
    'school_density']

df[distribution_features].hist(
    figsize=(12, 8),
    bins=25)

plt.suptitle('Distributions of Main Neighborhood Features')
plt.tight_layout()
plt.show()

- Walkability: widespread across tracts
- Climate risk: widespread across tracts
- Transit stop density: heavily skewed
- School density: heavily skewed, with most tracts near the lower end and a few very high values
- Social vulnerability: Clustered at the minimum -- needs a closer look
- School academic score: should look at this in a different way since it's more categorical


In [ ]:
# checking the largest values in the skewed features

skewed_features = [
    'transit_stop_density',
    'school_density',
    'social_vulnerability_score'
]

df[skewed_features].quantile(
    [0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1]
).round(2)